# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [1]:
#!pip install -qU ragas==0.2.10

In [2]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [3]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/jthomazo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/jthomazo/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [4]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [5]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [6]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - and download our webpages which we'll be using for our data today.

These webpages are from [Simon Willison's](https://simonwillison.net/) yearly "AI learnings".

- [2023 Blog](https://simonwillison.net/2023/Dec/31/ai-in-2023/)
- [2024 Blog](https://simonwillison.net/2024/Dec/31/llms-in-2024/)

Let's start by collecting our data into a useful pile!

In [7]:
!mkdir data

mkdir: data: File exists


In [8]:
!curl https://simonwillison.net/2023/Dec/31/ai-in-2023/ -o data/2023_llms.html

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 31493    0 31493    0     0  64814      0 --:--:-- --:--:-- --:--:-- 64800


In [9]:
!curl https://simonwillison.net/2024/Dec/31/llms-in-2024/ -o data/2024_llms.html

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 70519    0 70519    0     0   113k      0 --:--:-- --:--:-- --:--:--  113k


Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [10]:
from langchain_community.document_loaders import DirectoryLoader

path = "data/"
loader = DirectoryLoader(path, glob="*.html")
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [11]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [12]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [13]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [14]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/13 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/20 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 11, relationships: 27)

We can save and load our knowledge graphs as follows.

In [15]:
kg.save("ai_across_years_kg.json")
ai_across_years_kg = KnowledgeGraph.load("ai_across_years_kg.json")
ai_across_years_kg

KnowledgeGraph(nodes: 11, relationships: 27)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [16]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=ai_across_years_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [17]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.


#### 😎 ANSWER #1:

The 3 types of RAGAS query synthesizers used here are:

1. **Single-Hop Specific Query Synthesizer**:
This synthesizer generates simple, straightforward queries that require only one source of information (a single "hop"). It focuses on specific, factual information that can be found in a single document chunk.

2. **MultiHopAbstractQuerySynthesizer**:
This synthesizer creates more complex queries that require linking several different sources of information (multiple "hops"). These queries are more abstract and conceptual in nature, requiring a more general and theoretical understanding. They may require synthesizing information from different parts of a document or from multiple documents.

3. **MultiHopSpecificQuerySynthesizer**:
This synthesizer also generates queries requiring multiple sources of information, but remains more focused on specific facts than the abstract synthesizer. These queries require bringing together specific information from different sources to form a complete answer.


The distribution of generated queries will be :
- 25% of queries generated by the MultiHopAbstractQuerySynthesizer
- 25% of queries generated by the MultiHopSpecificQuerySynthesizer
- 50% of queries generated by the SingleHopSpecificQuerySynthesizer


Finally, we can use our `TestSetGenerator` to generate our testset!

In [18]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is the significance of the Falcon model i...,[The ethics of this space remain diabolically ...,"The Falcon model, developed by TII in Abu Dhab...",single_hop_specifc_query_synthesizer
1,Whaat is the role of AI in develpment of large...,[Simon Willison’s Weblog Subscribe Stuff we fi...,Simon Willison’s weblog notes that 2023 was a ...,single_hop_specifc_query_synthesizer
2,"As an AI researcher and developer, how does th...",[the document includes some of the clearest ex...,The document highlights ChatGPT as a prominent...,single_hop_specifc_query_synthesizer
3,What is GPT-4's significance according to the ...,[The rise of inference-scaling “reasoning” mod...,"The GPT-4 barrier was comprehensively broken, ...",single_hop_specifc_query_synthesizer
4,What is Google NotebookLM's recent advancement...,[you talk to me exclusively in Spanish. OpenAI...,"Google’s NotebookLM, released in September, to...",single_hop_specifc_query_synthesizer
5,Is it ok to train models on ppl's content with...,[<1-hop>\n\nThe ethics of this space remain di...,The document discusses the complex ethics of A...,multi_hop_abstract_query_synthesizer
6,how synthetic data helps train LLMs and why mo...,"[<1-hop>\n\n2024 was the year that the word ""s...",The context explains that synthetic data is us...,multi_hop_abstract_query_synthesizer
7,How do the legal and ethical considerations of...,[<1-hop>\n\nthe document includes some of the ...,The context discusses complex legal arguments ...,multi_hop_abstract_query_synthesizer
8,How does Amazon's recent announcement of voice...,[<1-hop>\n\nyou talk to me exclusively in Span...,The context indicates that Amazon pre-announce...,multi_hop_specific_query_synthesizer
9,How does the development of Llama 3.1 and Llam...,[<1-hop>\n\nThe rise of inference-scaling “rea...,The context highlights that models like Llama ...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [19]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

unable to apply transformation: Invalid json output: In 2024, advancements in Large Language Models included breaking the GPT-4 barrier with models surpassing it, increasing context lengths up to 2 million tokens, and widespread multimodal capabilities involving images, audio, and video. The cost of running LLMs decreased significantly due to competition and efficiency improvements, making prompts cheaper and reducing environmental impact, though infrastructure expansion by big tech raised concerns. Models like DeepSeek v3 demonstrated large-scale open licensing at a fraction of training costs, while inference-scaling models like OpenAI's o1 series enabled more complex reasoning through increased compute during inference. Accessibility to top-tier models was temporarily free but has since shifted to paid subscriptions, limiting free access. The concept of AI "agents" remains vague and unfulfilled, hindered by issues like prompt injection and over-reliance on gullible models. Evaluation

Applying CustomNodeFilter:   0%|          | 0/14 [00:00<?, ?it/s]

...

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

unable to apply transformation: Node 58ca4748-be59-4dfe-a137-aa8e1ccb2cd2 has no summary_embedding


Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [20]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Considering the advancements and current lands...,[My blog in 2023 Here’s the sequel to this pos...,"According to the provided context, OpenAI was ...",single_hop_specifc_query_synthesizer
1,What are the main challenges and developments ...,[openly licensed ones are still the world’s mo...,The research enthusiast highlights that openly...,single_hop_specifc_query_synthesizer
2,Whaat are the main challenges and ethical issu...,[Simon Willison’s Weblog Subscribe Stuff we fi...,Simon Willison’s weblog notes that the ethics ...,single_hop_specifc_query_synthesizer
3,What are large language models and how do they...,"[of what LLMs are, how they work and how they ...",Large language models are a significant topic ...,single_hop_specifc_query_synthesizer
4,Whaat is the EU AI act and how does it impact ...,[More recent articles AI assisted search-based...,Meta's Llama claims to be open source because ...,single_hop_specifc_query_synthesizer
5,How do the recent developments in multi-modal ...,[<1-hop>\n\non a story about the town's histor...,"The context highlights that in 2024, large lan...",multi_hop_specific_query_synthesizer
6,How does Apple’s MLX library improve LLM perfo...,[<1-hop>\n\nSimon Willison’s Weblog Subscribe ...,Apple’s MLX library enhances LLM performance b...,multi_hop_specific_query_synthesizer
7,Considering the developments in large language...,[<1-hop>\n\nopenly licensed ones are still the...,"The insights from September 2022, particularly...",multi_hop_specific_query_synthesizer
8,"H0w is Llama 3 b3ing ad0pted in AI researh, an...",[<1-hop>\n\nMore recent articles AI assisted s...,"Based on the context, recent articles mention ...",multi_hop_specific_query_synthesizer
9,"how Claude and Claude like, do they both good ...",[<1-hop>\n\nThe rise of inference-scaling “rea...,"Based on the context, Claude refers to a serie...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [21]:
from langsmith import Client

client = Client()

dataset_name = "State of AI Across the Years!"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="State of AI Across the Years!"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [22]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [25]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [26]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

In [28]:
rag_documents[:5]

[Document(metadata={'source': 'data/2023_llms.html'}, page_content='Simon Willison’s Weblog\n\nSubscribe\n\nStuff we figured out about AI in 2023\n\n31st December 2023\n\n2023 was the breakthrough year for Large Language Models (LLMs). I think it’s OK to call these AI—they’re the latest and (currently) most interesting development in the academic field of Artificial Intelligence that dates back to the 1950s.\n\nHere’s my attempt to round up the highlights in one place!\n\nLarge Language Models\n\nThey’re actually quite easy to build\n\nYou can run LLMs on your own devices'),
 Document(metadata={'source': 'data/2023_llms.html'}, page_content='You can run LLMs on your own devices\n\nHobbyists can build their own fine-tuned models\n\nWe don’t yet know how to build GPT-4\n\nVibes Based Development\n\nLLMs are really smart, and also really, really dumb\n\nGullibility is the biggest unsolved problem\n\nCode may be the best application\n\nThe ethics of this space remain diabolically complex\n

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [29]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [30]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="State of AI"
)

/Users/jthomazo/Archives/01_Projets/02_AIM/AIE6/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/qdrant_client/http/models/models.py:758: SyntaxWarning: invalid escape sequence '\&'
  description="Check that the field is empty, alternative syntax for `is_empty: \&quot;field_name\&quot;`",
/Users/jthomazo/Archives/01_Projets/02_AIM/AIE6/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/qdrant_client/http/models/models.py:762: SyntaxWarning: invalid escape sequence '\&'
  description="Check that the field is null, alternative syntax for `is_null: \&quot;field_name\&quot;`",


In [31]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [32]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [33]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [34]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [35]:
rag_chain.invoke({"question" : "What are Agents?"})

'Based on the provided context, "agents" is an infuriatingly vague and poorly defined term related to AI. The term "agents" generally refers to AI systems that can go away and act on your behalf, similar to the concept of a travel agent acting for you or large language models (LLMs) given access to tools which they can use in a loop to solve problems. However, there is no single, clear, and widely understood meaning of "agents," and definitions vary widely.\n\nThe difficulty in pinning down what "agents" are is compounded by challenges such as gullibility (LLMs believing anything told to them) and the lack of fully realized examples running in production. Some think achieving robust agents depends on achieving AGI (Artificial General Intelligence). Despite much discussion and numerous prototypes, true AI agents remain largely a "coming soon" concept rather than a present reality.\n\nIn summary, agents are AI systems intended to act autonomously on a user\'s behalf, but the concept rema

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [36]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [37]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dope_or_nope_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this submission dope, lit, or cool?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `dope_or_nope_evaluator`:

### 😎 ANSWER #2:
1. **"qa_evaluator"** evaluates the factual correctness of answers against a reference. It is a standard "qa" evaluator of LangSmith which checks if the answer generated by the system is correct and accurate with respect to the question posed.

2. **"labeled_helpfulness_evaluator"** evaluates the helpfulness of answers to the user, taking into account the reference answer.
It uses the  "labeled_criteria" evaluator of  LangSmith with the specific criterion "helpfulness." It examines if the generated answer actually helps the user solve their problem or answer their question, by comparing it with the reference answer provided in the dataset.

3. **"dope_or_nope_evaluator"** evaluates whether the answer is "cool," "dope," or "lit" based on stylistic criteria. 
It uses the  "criteria" evaluator of  LangSmith with the "dopeness" criterion. This evaluator focuses on the stylistic aspect and tone of the answer which can be important for user experience.

## LangSmith Evaluation

In [38]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dope_or_nope_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'loyal-minute-29' at:
https://smith.langchain.com/o/0f3983de-eb9f-4417-bd1b-e1b02dc03750/datasets/e7dba11c-74d1-472b-a6e5-c903381d25cf/compare?selectedSessions=7a4bd034-603b-4632-a82e-cba441580a93




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,"how Claude and Claude like, do they both good ...","Based on the provided context, Claude models, ...",None,"Based on the context, Claude refers to a serie...",1,1,0,4.562608,7552eca9-c961-4fcd-907e-a8895e60ed32,041633d4-e575-4844-85b9-f2f8b6d9dba7
1,"H0w is Llama 3 b3ing ad0pted in AI researh, an...","Based on the provided context, Llama 3 is bein...",None,"Based on the context, recent articles mention ...",0,0,0,2.737478,57d47feb-822e-42b6-8d9a-4acc3b7fbb53,15e6f910-90d2-401e-b325-d34d80849355
2,Considering the developments in large language...,"Based on the provided context, in September 20...",None,"The insights from September 2022, particularly...",1,0,0,4.381025,49c2194a-7941-4b3e-8676-2db3890d9b5a,3bb9f31e-7a87-4166-936b-029d01a77d59
3,How does Apple’s MLX library improve LLM perfo...,Apple’s MLX library improves LLM performance p...,None,Apple’s MLX library enhances LLM performance b...,1,0,0,3.932263,e879cd07-ae3a-4dc6-8bc5-7035a31aa4c9,0f74f1ea-67d2-4454-ba8e-2a6f3212520e
4,How do the recent developments in multi-modal ...,"Based on the provided context, recent multi-mo...",None,"The context highlights that in 2024, large lan...",1,1,0,4.820302,46f26582-1420-4261-a797-512ce6085987,98013c7d-6625-4735-bdde-59eb1e65390d
5,Whaat is the EU AI act and how does it impact ...,"Based on the provided context, the EU AI act i...",None,Meta's Llama claims to be open source because ...,1,0,0,3.173533,0ac286e5-cfde-4d80-bba1-01224ee0ef4f,70d9f698-e1eb-49e7-a276-28cba7906207
6,What are large language models and how do they...,Large Language Models (LLMs) are software crea...,None,Large language models are a significant topic ...,1,0,0,2.980538,8b08014a-b060-44a1-9c8e-01329c92c9a8,a11f0f43-93cd-478b-ae2b-08a7042b871a
7,Whaat are the main challenges and ethical issu...,The main challenges and ethical issues related...,None,Simon Willison’s weblog notes that the ethics ...,1,1,0,3.692084,e7b8e887-cac2-4420-9246-5c7c96435a77,897dfb70-4702-4faa-a9b6-d841ce6ffecb
8,What are the main challenges and developments ...,"According to the research enthusiast, the main...",None,The research enthusiast highlights that openly...,0,0,0,3.704799,6a91861c-2e4f-479e-8832-5a1ee323e8b4,54ce54a4-e49f-4d2d-b47e-754cfef7234b
9,Considering the advancements and current lands...,"Based on the provided context, OpenAI has play...",None,"According to the provided context, OpenAI was ...",1,1,0,4.197388,81f97991-4939-45bb-a59e-6218e901c327,a880dabe-19d6-4954-910d-4024a90a6319


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [39]:
DOPE_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the questions in a dope way, be cool!

Context: {context}
Question: {question}
"""

dope_rag_prompt = ChatPromptTemplate.from_template(DOPE_RAG_PROMPT)

In [40]:
rag_documents = docs

In [41]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

In [64]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

In [42]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="AI Across Years (Augmented)"
)

In [43]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [44]:
dope_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dope_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [45]:
dope_rag_chain.invoke({"question" : "what are Agents?"})

"Yo, agents are kinda like the mysterious cool kids of AI right now — folks throw around the term, but no one’s nailed down one clean meaning. Some peep them as AI that straight-up acts on your behalf, like a travel agent hustling for you. Others see agents as large language models flexing with tools in a loop to solve problems, kinda like AI multitasking on your tasks.\n\nBut here’s the kicker — “agents” still feel like they’re coming soon, and their vibe is tangled with things like “autonomy,” which people throw in without clear definitions. Plus, these agents struggle big time with gullibility — they tend to believe everything they're told, which makes real-world decision-making tricky. Without a fix on that, agents are more like prototypes than fully reliable players.\n\nSo, in short: Agents = AI systems designed to act on your behalf, but the concept is still fuzzy and waiting for the tech (and AGI-level smarts) to back it up. Cool but complicated, ya feel me?"

Finally, we can evaluate the new chain on the same test set!

In [46]:
evaluate(
    dope_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dope_or_nope_evaluator
    ],
    metadata={"revision_id": "dope_chain"},
)

View the evaluation results for experiment: 'giving-jewel-11' at:
https://smith.langchain.com/o/0f3983de-eb9f-4417-bd1b-e1b02dc03750/datasets/e7dba11c-74d1-472b-a6e5-c903381d25cf/compare?selectedSessions=f4be3626-6acb-4938-b679-d5a870ab1ee4




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,"how Claude and Claude like, do they both good ...","Oh, you’re talking about Claude and its whole ...",None,"Based on the context, Claude refers to a serie...",1,1,1,2.647895,7552eca9-c961-4fcd-907e-a8895e60ed32,0b95c02f-dd8c-49fe-a5c8-a02406193270
1,"H0w is Llama 3 b3ing ad0pted in AI researh, an...","Alright, here’s the lowdown on Llama 3 in the ...",None,"Based on the context, recent articles mention ...",1,0,1,4.402169,57d47feb-822e-42b6-8d9a-4acc3b7fbb53,f0e510cf-24e4-454f-91f8-33a8c16f94a9
2,Considering the developments in large language...,"Yo, here’s the lowdown, straight from the AI s...",None,"The insights from September 2022, particularly...",1,1,1,3.519061,49c2194a-7941-4b3e-8676-2db3890d9b5a,bb6e6895-2e2a-4c52-803b-4b253c5f4a8e
3,How does Apple’s MLX library improve LLM perfo...,"Alright, here’s the lowdown, fresh and fly:\n\...",None,Apple’s MLX library enhances LLM performance b...,1,1,1,5.809415,e879cd07-ae3a-4dc6-8bc5-7035a31aa4c9,bcea4aae-4f91-43b7-9022-84c4c04c3331
4,How do the recent developments in multi-modal ...,"Yo, here’s the lowdown: In 2024, multi-modal m...",None,"The context highlights that in 2024, large lan...",1,1,1,5.399591,46f26582-1420-4261-a797-512ce6085987,2d5f6e2d-3a0b-483e-a6be-29958d9b3d21
5,Whaat is the EU AI act and how does it impact ...,"Yo, here’s the scoop straight from the context...",None,Meta's Llama claims to be open source because ...,1,1,1,2.996575,0ac286e5-cfde-4d80-bba1-01224ee0ef4f,20110c47-b26c-4fc0-8a3e-81e47d0adfe2
6,What are large language models and how do they...,"Alright, here’s the scoop straight from the AI...",None,Large language models are a significant topic ...,0,0,1,3.638742,8b08014a-b060-44a1-9c8e-01329c92c9a8,d2fc1dac-bae8-4073-920e-ed9f9c67d382
7,Whaat are the main challenges and ethical issu...,"Yo, here’s the lowdown on the main challenges ...",None,Simon Willison’s weblog notes that the ethics ...,0,0,1,4.109967,e7b8e887-cac2-4420-9246-5c7c96435a77,ed343195-e2bf-4285-b0f8-a30f0ea2dd2b
8,What are the main challenges and developments ...,"Yo, here’s the lowdown on 2024 vibes from the ...",None,The research enthusiast highlights that openly...,1,0,1,6.823725,6a91861c-2e4f-479e-8832-5a1ee323e8b4,40e56b3f-dc74-4091-b337-c0cc285021af
9,Considering the advancements and current lands...,"Alright, check this out—OpenAI has been the he...",None,"According to the provided context, OpenAI was ...",0,0,1,4.544154,81f97991-4939-45bb-a59e-6218e901c327,4480a2a6-f3f6-4920-8f97-12e52549b9d0


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.